In [20]:
!pip install --upgrade pip setuptools wheel -q
!pip install --upgrade cmake -q
!pip install scs --prefer-binary -q
!pip install cvxpy --prefer-binary -q

In [21]:
!pip install awswrangler -q
!pip install optbinning -q
!pip install lightgbm
!pip install xgboost
!pip install xgboost --prefer-binary
!pip install catboost

In [22]:
import awswrangler as wr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from optbinning import BinningProcess
import shutil
from warnings import simplefilter
simplefilter(action = "ignore") #, category = FutureWarning

pd.set_option('display.max_rows', 500)
from sklearn.preprocessing import LabelEncoder

In [23]:
import pandas as pd
import numpy as np
import seaborn as sb
import matplotlib.pyplot as plt
from typing import List, Tuple
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import plotly.express as px

pd.set_option('display.float_format', '{:.2f}'.format)

In [25]:
%%time
query = """
WITH pd AS (
    SELECT DISTINCT
        -- Identificadores
        key_value,
        cod_cli,
        date_format(date_parse(CAST(cod_mes AS varchar), '%Y%m') - interval '1' month, '%Y%m') AS codmes_lag1,
        CAST(cod_mes AS INTEGER) AS cod_mes,

        -- Fechas
        TRY_CAST(fec_constitucion AS DATE) AS fecha_constitucion,

        -- Monetarios (mantener nombres originales)
        TRY_CAST(mto_pas_soles AS DOUBLE) AS mto_pas_soles,
        TRY_CAST(imp_trx_abonosefect_6m AS DOUBLE) AS imp_trx_abonosefect_6m,
        TRY_CAST(imp_trx_cargosefe_6m AS DOUBLE) AS imp_trx_cargosefe_6m,
        TRY_CAST(avg_trx_cargostot_3m AS DOUBLE) AS avg_trx_cargostot_3m,
        TRY_CAST(max_trx_abonos_3m AS DOUBLE) AS max_trx_abonos_3m,

        -- Cantidades
        TRY_CAST(cnt_trx_cargostot_3m AS INTEGER) AS cnt_trx_cargostot_3m,

        -- Promedios / ratios
        TRY_CAST(cnt_trx_abonospromtot_3m AS DOUBLE) AS cnt_trx_abonospromtot_3m,
        TRY_CAST(rat_trx_abonosefectot_1m AS DOUBLE) AS rat_trx_abonosefectot_1m,
        TRY_CAST(rat_trx_abonosefectot_3m AS DOUBLE) AS rat_trx_abonosefectot_3m,
        TRY_CAST(rat_trx_abonosefectot_9m AS DOUBLE) AS rat_trx_abonosefectot_9m,
        TRY_CAST(rat_mntcrgsefetot_1m AS DOUBLE) AS rat_mntcrgsefetot_1m,

        -- Demográficas / antigüedad
        TRY_CAST(num_edad_constitucion AS INTEGER) AS num_edad_constitucion,
        TRY_CAST(num_antiguedad AS INTEGER) AS num_antiguedad,

        -- Riesgo
        TRY_CAST(desc_nivel_rsg_lsb_tot AS DOUBLE) AS desc_nivel_rsg_lsb_tot,

        -- Actividad mensual
        TRY_CAST(cnt_meses_siningresos_12m AS INTEGER) AS cnt_meses_siningresos_12m,
        TRY_CAST(cnt_meses_sinegresos_12m AS INTEGER) AS cnt_meses_sinegresos_12m,

        -- Ubicación / segmentación
        desc_provincia AS desc_provincia,
        desc_departamento AS desc_departamento,
        cod_ubigeo_cd AS cod_ubigeo_cd,
        cod_sectorista_id AS cod_sectorista_id,
        cod_ciiu_v4 AS cod_ciiu_v4,

        -- Flags (string/bool → 0/1)
        flg_casos_hist,
        flg_vrcn_abonos_5m_1m ,
        flg_vrcn_efe_cargos_5m_1m ,

        -- Conteos
        cnt_ro_debajo_umbral AS cnt_ro_debajo_umbral,

        -- Perfil económico
        mto_fact_declarado_sunat AS mto_fact_declarado_sunat,
        TRY_CAST(avg_cp_men_ing_12m AS DOUBLE) AS avg_cp_men_ing_12m,
        TRY_CAST(avg_cpmenegr_12m AS DOUBLE) AS avg_cpmenegr_12m,
        TRY_CAST(max_mto_cpmening_12m AS DOUBLE) AS max_mto_cpmening_12m,
        TRY_CAST(max_mto_cpegrmen_12m AS DOUBLE) AS max_mto_cpegrmen_12m,

        -- Exterior
        flg_al_ext_12m AS flg_al_ext_12m,
        flg_del_ext_12m AS flg_del_ext_12m,
        TRY_CAST(cnt_trx_sinenv_alext_12m AS INTEGER) AS cnt_trx_sinenv_alext_12m,
        TRY_CAST(cnt_trx_al_ext_1000_12m AS INTEGER) AS cnt_trx_al_ext_1000_12m,
        TRY_CAST(mto_al_ext_12m AS DOUBLE) AS mto_al_ext_12m,
        TRY_CAST(mto_del_ext_12m AS DOUBLE) AS mto_del_ext_12m,

        -- Reputacional / antecedentes
        flg_pep AS flg_pep,
        cod_rsg_pep AS cod_rsg_pep,
        flg_activo_pep AS flg_activo_pep,
        TRY_CAST(cnt_noticias AS INTEGER) AS cnt_noticias,
        flg_ros_12m AS flg_ros_12m,
        flg_alerta_12m AS flg_alerta_12m,
        TRY_CAST(cnt_alerta_hist AS INTEGER) AS cnt_alerta_hist,
        TRY_CAST(cnt_ros_hist AS INTEGER) AS cnt_ros_hist,

        -- KYC
        flg_kyc_12m AS flg_kyc_12m,
        flg_kyc_hist AS flg_kyc_hist,
        TRY_CAST(cnt_kyc_hist AS INTEGER) AS cnt_kyc_hist,

        -- ======================================================
        -- 🔹 ACELERACIÓN / CAMBIO DE COMPORTAMIENTO
        -- ======================================================
        TRY_CAST(imp_trx_abonostot_1m AS DOUBLE) / NULLIF(TRY_CAST(avg_trx_abonostot_6m AS DOUBLE), 0) AS ratio_abonos_1m_vs_6m,
        TRY_CAST(imp_trx_cargostot_1m AS DOUBLE) / NULLIF(TRY_CAST(avg_trx_cargostot_6m AS DOUBLE), 0) AS ratio_cargos_1m_vs_6m,

        -- ======================================================
        -- 🔹 CONCENTRACIÓN EN CONTRAPARTE
        -- ======================================================
        TRY_CAST(avg_cpmenegr_12m AS DOUBLE) / NULLIF(TRY_CAST(imp_trx_cargostot_6m AS DOUBLE), 0) AS share_cp_egresos,
        TRY_CAST(avg_cp_men_ing_12m AS DOUBLE) / NULLIF(TRY_CAST(imp_trx_abonostot_6m AS DOUBLE), 0) AS share_cp_ingresos,

        -- ======================================================
        -- 🔹 NORMALIZACIÓN DE RIESGO
        -- ======================================================
        TRY_CAST(cnt_ros_hist AS DOUBLE) / NULLIF(TRY_CAST(cnt_trx_cargostot_3m AS DOUBLE), 0) AS ros_por_trx_3m,
        TRY_CAST(cnt_alerta_hist AS DOUBLE) / NULLIF(TRY_CAST(num_antiguedad AS DOUBLE), 0) AS alertas_por_antiguedad,

        -- ======================================================
        -- 🔹 COHERENCIA ECONÓMICA
        -- ======================================================
        TRY_CAST(imp_trx_abonostot_6m AS DOUBLE) / NULLIF(TRY_CAST(mto_fact_declarado_sunat AS DOUBLE), 0) AS ingresos_vs_facturacion,
        TRY_CAST(mto_pas_soles AS DOUBLE) / NULLIF(TRY_CAST(imp_trx_abonostot_6m AS DOUBLE), 0) AS pasivo_vs_ingresos,

        -- ======================================================
        -- 🔹 EXPOSICIÓN AL EXTERIOR (PROPORCIONES)
        -- ======================================================
        TRY_CAST(mto_al_ext_12m AS DOUBLE) / NULLIF(TRY_CAST(imp_trx_cargostot_12m AS DOUBLE), 0) AS ratio_egresos_exterior,
        TRY_CAST(mto_del_ext_12m AS DOUBLE) / NULLIF(TRY_CAST(imp_trx_abonostot_12m AS DOUBLE), 0) AS ratio_ingresos_exterior,

        -- ======================================================
        -- 🔹 COHERENCIA PEP / LSB
        -- ======================================================
        TRY_CAST(cod_rsg_pep AS DOUBLE) - TRY_CAST(desc_nivel_rsg_lsb_tot AS DOUBLE) AS gap_riesgo_pep_lsb

    FROM d_perm_aws.t_agg_alertas_plaft_hist_bkp
    WHERE cod_mes = '202602'
    AND desc_subsegmento  = 'Renta Alta'
),

target AS (
    SELECT 
        codunico,
        periodo_alerta,
        tipo_alerta_n2 ,
        MAX(calificacion_monitoreo) AS flg_alerta
    FROM e_perm_aws.t_alertas_plaft
    GROUP BY codunico, periodo_alerta, tipo_alerta_n2
),

-- =========================
-- 360 CLIENTE
-- =========================
pd_02 AS (
    SELECT
        key_value,
        codmes,
        MAX(edad) AS edad,
        MAX(ingreso_bruto) AS ingreso_bruto
    FROM e_perm_aws.v_aws_360_cliente_dia
    WHERE flg_fallecido = 'N'
      AND cod_tipo_documento = 1
      AND CAST(codmes AS INTEGER) BETWEEN 202601 AND 202604
    GROUP BY key_value, codmes
),

-- =========================
-- RCC AGG01
-- =========================
pd_05 AS (
    SELECT
        key_value,
        p_codmes,
        prom_lin_tc_rccsf_06m,
        lin_tcrrstsf03m,
        prm_usotcrrstsf03m,
        prm_lintcrallsf12m,
        prm_lintcrrstsf03m,
        cre_saltot_tc_rccsf_m02,
        prom_salvig_entprinc_tc_rccsf_03m,
        cre_salvig_tc_rccsf_m02,
        ind_max_salvig_tc_rccsf_06m,
        var_usotcrrstsf03m,
        ctd_prod_rccsf_m01,
        ind_min_salvig_tc_rccsf_06m,
        lintot_tc_rccsf_03m,
        prom_salvig_pp_rccsf_06m,
        salvig_pp_rccsf_06m,
        cre_pct_salvig_tc_rccsf_m03,
        prom_salvig_tc_rccsf_06m,
        cre_lin_tc_rccsf_m02,
        var_lintcrrstsf03m
    FROM e_perm_aws.tbl_rcc_agg01_allsf_mdl
    WHERE CAST(p_codmes AS INTEGER) BETWEEN 202601 AND 202604
),

-- =========================
-- PERFIL
-- =========================
pd_06 AS (
    SELECT
        key_value,
        p_codmes,
        rgn,
        tip_lvledu
    FROM e_perm_aws.tbl_per_inf_mdl
    WHERE CAST(p_codmes AS INTEGER) BETWEEN 202601 AND 202604
),

-- =========================
-- RIESGO
-- =========================
pd_07 AS (
    SELECT
        key_value,
        p_codmes,
        ing_brt,
        ind_lin_ing_tcr_ibk,
        cem
    FROM e_perm_aws.tbl_rsk_inf_mdl
    WHERE CAST(p_codmes AS INTEGER) BETWEEN 202601 AND 202604
),

-- =========================
-- RCC AGG02
-- =========================
pd_08 AS (
    SELECT
        key_value,
        p_codmes,
        AVG(sow_lnep1tcrallsfm01) AS sow_lnep1tcrallsfm01,
        AVG(sow_svep1actallsfm01) AS sow_svep1actallsfm01
    FROM e_perm_aws.tbl_rcc_agg02_allsf_mdl
    WHERE CAST(p_codmes AS INTEGER) BETWEEN 202601 AND 202604
    GROUP BY key_value, p_codmes
),

-- =========================
-- CAMPAÑAS
-- =========================
pd_09 AS (
    SELECT
        key_value,
        p_codmes,
        ctd_camptot06m,
        prm_camptot06m,
        max_camptot06m,
        min_camptot06m,
        rec_camptot06m
    FROM e_perm_aws.tbl_cmp_tot_inf_mdl
    WHERE tip_doc = '1'
      AND CAST(p_codmes AS INTEGER) BETWEEN 202601 AND 202604
),

-- =========================
-- CANALES
-- =========================
pd_10 AS (
    SELECT
        key_value,
        mes AS p_codmes,
        atm_monto,
        atm_frec,
        atm_recen,
        atm_trx_prom,
        atm_trx_ret_prom,
        atm_trx_dep,
        atm_trx_dep_prom,
        atm_trx_nmon_prom,
        atm_trx_nmon_prom2,
        atm_trx_nmon_min,
        atm_trx_con,
        atm_trx_con_prom,
        bpi_monto,
        bpi_trx_prom,
        bpi_trx_prom2,
        bpi_trx_nmon_prom,
        bpi_trx_nmon_prom2,
        bpi_trx_mon_prom,
        bpi_trx_mon_prom2,
        bpi_trx_mon_min,
        bpi_trx_mon_max,
        bpi_recen_mon,
        bpi_recen_nmon,
        grupo_final
    FROM e_perm_aws.v_seg_canales
    WHERE tipodocumento = '1'
      AND CAST(mes AS INTEGER) BETWEEN 202601 AND 202604
),

-- =========================
-- EDUCACIÓN (CORREGIDO)
-- =========================
pd_11 AS (
    SELECT
        key_value,
        p_codmes,
        lvl_edu
    FROM e_perm_aws.t_sentinel_rsk
    WHERE CAST(p_codmes AS INTEGER) BETWEEN 202601 AND 202604
)

-- =========================
-- SELECT FINAL
-- =========================
SELECT
    -- IDs
    CAST(a.cod_mes AS VARCHAR) AS cod_mes,
    a.cod_cli,
    a.key_value AS key_value,

    -- Control
    CASE WHEN b.flg_alerta = '1' THEN 1 ELSE 0 END AS target_m,
    b.tipo_alerta_n2,

    -- Variables del modelo (COLS_VARS) -> referenciar nombres originales y mantener alias de salida
    a.mto_pas_soles AS mto_pas_soles,
    a.ros_por_trx_3m,
    a.alertas_por_antiguedad,
    c.edad,
    a.ratio_abonos_1m_vs_6m,
    k.bpi_trx_mon_min,
    i.sow_svep1actallsfm01,
    k.grupo_final,
    a.imp_trx_cargosefe_6m AS imp_trx_cargosefe_6m,
    a.pasivo_vs_ingresos,
    i.sow_lnep1tcrallsfm01,
    a.rat_trx_abonosefectot_3m AS rat_trx_abonosefectot_3m,
    k.atm_monto,
    h.ind_lin_ing_tcr_ibk,
    k.atm_recen,
    a.desc_provincia AS desc_provincia,
    a.cnt_trx_abonospromtot_3m AS cnt_trx_abonospromtot_3m,
    a.cnt_alerta_hist AS cnt_alerta_hist,
    a.flg_alerta_12m AS flg_alerta_12m,
    a.num_antiguedad AS num_antiguedad,
    a.rat_trx_abonosefectot_9m AS rat_trx_abonosefectot_9m,
    f.prm_usotcrrstsf03m,
    a.cod_ubigeo_cd AS cod_ubigeo_cd,
    f.var_lintcrrstsf03m,
    m.lvl_edu,
    a.imp_trx_abonosefect_6m AS imp_trx_abonosefect_6m,
    a.cnt_trx_cargostot_3m AS cnt_trx_cargostot_3m,
    f.cre_pct_salvig_tc_rccsf_m03,
    j.ctd_camptot06m,
    a.desc_departamento AS desc_departamento,
    a.ratio_cargos_1m_vs_6m,
    a.avg_trx_cargostot_3m AS avg_trx_cargostot_3m,
    f.cre_saltot_tc_rccsf_m02,
    a.cnt_meses_siningresos_12m AS cnt_meses_siningresos_12m,
    f.prom_salvig_pp_rccsf_06m,
    a.cnt_trx_sinenv_alext_12m AS cnt_trx_sinenv_alext_12m,
    k.atm_frec,
    f.cre_salvig_tc_rccsf_m02,
    j.prm_camptot06m,
    a.cnt_meses_sinegresos_12m AS cnt_meses_sinegresos_12m,
    f.var_usotcrrstsf03m,
    j.rec_camptot06m,
    a.cod_rsg_pep AS cod_rsg_pep,
    a.flg_pep AS flg_pep,
    j.max_camptot06m

FROM pd a
LEFT JOIN target b ON a.cod_cli = b.codunico AND CAST(a.cod_mes AS VARCHAR) = b.periodo_alerta
LEFT JOIN pd_02 c ON CAST(a.codmes_lag1 AS VARCHAR) = c.codmes AND a.key_value = c.key_value
LEFT JOIN pd_05 f ON CAST(a.codmes_lag1 AS VARCHAR) = f.p_codmes AND a.key_value = f.key_value
LEFT JOIN pd_07 h ON CAST(a.codmes_lag1 AS VARCHAR) = h.p_codmes AND a.key_value = h.key_value
LEFT JOIN pd_08 i ON CAST(a.codmes_lag1 AS VARCHAR) = i.p_codmes AND a.key_value = i.key_value
LEFT JOIN pd_09 j ON CAST(a.codmes_lag1 AS VARCHAR) = j.p_codmes AND a.key_value = j.key_value
LEFT JOIN pd_10 k ON CAST(a.codmes_lag1 AS VARCHAR) = k.p_codmes AND a.key_value = k.key_value
LEFT JOIN pd_11 m ON CAST(a.codmes_lag1 AS VARCHAR) = m.p_codmes AND a.key_value = m.key_value
"""
df_dataset = wr.athena.read_sql_query(query, database = 'disc_comercial', ctas_approach=False)
df_dataset.head()

CPU times: user 11.2 s, sys: 730 ms, total: 11.9 s
Wall time: 56.5 s


,cod_mes,cod_cli,key_value,target_m,tipo_alerta_n2,mto_pas_soles,ros_por_trx_3m,alertas_por_antiguedad,edad,ratio_abonos_1m_vs_6m,bpi_trx_mon_min,sow_svep1actallsfm01,grupo_final,imp_trx_cargosefe_6m,pasivo_vs_ingresos,sow_lnep1tcrallsfm01,rat_trx_abonosefectot_3m,atm_monto,ind_lin_ing_tcr_ibk,atm_recen,desc_provincia,cnt_trx_abonospromtot_3m,cnt_alerta_hist,flg_alerta_12m,num_antiguedad,rat_trx_abonosefectot_9m,prm_usotcrrstsf03m,cod_ubigeo_cd,var_lintcrrstsf03m,lvl_edu,imp_trx_abonosefect_6m,cnt_trx_cargostot_3m,cre_pct_salvig_tc_rccsf_m03,ctd_camptot06m,desc_departamento,ratio_cargos_1m_vs_6m,avg_trx_cargostot_3m,cre_saltot_tc_rccsf_m02,cnt_meses_siningresos_12m,prom_salvig_pp_rccsf_06m,cnt_trx_sinenv_alext_12m,atm_frec,cre_salvig_tc_rccsf_m02,prm_camptot06m,cnt_meses_sinegresos_12m,var_usotcrrstsf03m,rec_camptot06m,cod_rsg_pep,flg_pep,max_camptot06m
0,202602,0009414564,CEF48C576BAA57AFD8EC47311348D992DAB88E84E5C43D0A61F753A29FFF94AC,0,<NA>,1972.07,NaN,NaN,48,0.19,1,0.848,Digital,0.00,0.12,0.818,0.00,2472.400000000,0.000,6,LIMA,6.67,<NA>,0,21,0.00,0.396,150140,0.000,<NA>,0.00,44,0.335,11,LIMA,NaN,0.00,0.488,0,0.000,<NA>,2,0.488,1.833,0,-0.567,6,0,0,3
1,202602,0015184133,8EAFC45206B8420E139253386B72F3AE4D09795611EAC129F2CA212607284F8D,0,<NA>,20927.81,NaN,NaN,42,0.00,1,0.691,Digital,1150.00,0.52,0.528,0.00,700.000000000,0.000,1,LIMA,1.33,<NA>,0,8,0.00,0.759,150140,0.000,SECUNDARIA COMPLETA,0.00,18,0.002,31,LIMA,6145.47,0.33,0.009,0,20528.903,<NA>,1,0.009,5.167,0,-0.008,6,0,1,7
2,202602,0014786649,DB4BF756C539FAFA8BB074A91CE9CC6AD891F782890E9E63C274D5FA79917F67,0,<NA>,4207.36,NaN,NaN,34,1.00,1,1.000,Digital,50.00,0.04,0.000,0.00,470.000000000,0.000,1,LIMA,10.00,<NA>,0,9,0.00,0.000,150135,0.000,SECUNDARIA COMPLETA,0.00,111,0.000,25,LIMA,71576.23,0.00,0.000,0,0.000,<NA>,4,0.000,4.167,0,0.000,6,0,1,5
3,202602,0014607544,294871FE43C1D3E5154E31B1AD7E21652EE6A0D27EC2DA152F910F7FC379C728,0,<NA>,2615.07,NaN,NaN,33,0.10,2,0.735,Digital,1900.00,0.09,0.302,0.43,26546.360000000,0.000,2,LIMA,4.67,<NA>,0,9,0.28,0.209,150113,0.002,SUPERIOR 2º AÑO,13299.45,20,0.000,22,LIMA,5923.06,0.00,0.011,0,37277.890,<NA>,9,0.011,3.667,0,0.735,6,0,0,5
4,202602,0011399744,7A7C85FCF72CC3379E3560DA10BD2EA29771C932A91B323FC76B179898CA8414,0,<NA>,1591.06,NaN,NaN,78,0.44,1,0.000,Digital,8279.20,0.04,0.493,0.17,32400.000000000,0.000,1,LIMA,2.00,<NA>,0,16,0.25,0.018,150140,0.002,SUPERIOR COMPLETA,15980.00,43,-1.000,4,LIMA,4019.19,1.33,None,0,0.000,<NA>,6,None,2.000,0,1.000,3,<NA>,0,3


In [26]:
df_dataset.shape

(238065, 50)

In [27]:
df= df_dataset

In [28]:
# Opción recomendada - la más práctica
pd.set_option('display.max_columns', None)   # muestra todas las columnas
pd.set_option('display.width', 1000)         # evita que se corte horizontalmente
pd.set_option('display.max_colwidth', 100)   # si hay columnas con texto largo

# Ahora sí
print(df.columns)
# o mejor aún:
df.columns.tolist()   # lista limpia y completa

Index(['cod_mes', 'cod_cli', 'key_value', 'target_m', 'tipo_alerta_n2', 'mto_pas_soles', 'ros_por_trx_3m', 'alertas_por_antiguedad', 'edad', 'ratio_abonos_1m_vs_6m', 'bpi_trx_mon_min', 'sow_svep1actallsfm01', 'grupo_final', 'imp_trx_cargosefe_6m', 'pasivo_vs_ingresos', 'sow_lnep1tcrallsfm01', 'rat_trx_abonosefectot_3m', 'atm_monto', 'ind_lin_ing_tcr_ibk', 'atm_recen', 'desc_provincia', 'cnt_trx_abonospromtot_3m', 'cnt_alerta_hist', 'flg_alerta_12m', 'num_antiguedad', 'rat_trx_abonosefectot_9m', 'prm_usotcrrstsf03m', 'cod_ubigeo_cd', 'var_lintcrrstsf03m', 'lvl_edu', 'imp_trx_abonosefect_6m', 'cnt_trx_cargostot_3m', 'cre_pct_salvig_tc_rccsf_m03', 'ctd_camptot06m', 'desc_departamento', 'ratio_cargos_1m_vs_6m', 'avg_trx_cargostot_3m', 'cre_saltot_tc_rccsf_m02', 'cnt_meses_siningresos_12m', 'prom_salvig_pp_rccsf_06m', 'cnt_trx_sinenv_alext_12m', 'atm_frec', 'cre_salvig_tc_rccsf_m02', 'prm_camptot06m', 'cnt_meses_sinegresos_12m', 'var_usotcrrstsf03m', 'rec_camptot06m', 'cod_rsg_pep',
       

['cod_mes',
 'cod_cli',
 'key_value',
 'target_m',
 'tipo_alerta_n2',
 'mto_pas_soles',
 'ros_por_trx_3m',
 'alertas_por_antiguedad',
 'edad',
 'ratio_abonos_1m_vs_6m',
 'bpi_trx_mon_min',
 'sow_svep1actallsfm01',
 'grupo_final',
 'imp_trx_cargosefe_6m',
 'pasivo_vs_ingresos',
 'sow_lnep1tcrallsfm01',
 'rat_trx_abonosefectot_3m',
 'atm_monto',
 'ind_lin_ing_tcr_ibk',
 'atm_recen',
 'desc_provincia',
 'cnt_trx_abonospromtot_3m',
 'cnt_alerta_hist',
 'flg_alerta_12m',
 'num_antiguedad',
 'rat_trx_abonosefectot_9m',
 'prm_usotcrrstsf03m',
 'cod_ubigeo_cd',
 'var_lintcrrstsf03m',
 'lvl_edu',
 'imp_trx_abonosefect_6m',
 'cnt_trx_cargostot_3m',
 'cre_pct_salvig_tc_rccsf_m03',
 'ctd_camptot06m',
 'desc_departamento',
 'ratio_cargos_1m_vs_6m',
 'avg_trx_cargostot_3m',
 'cre_saltot_tc_rccsf_m02',
 'cnt_meses_siningresos_12m',
 'prom_salvig_pp_rccsf_06m',
 'cnt_trx_sinenv_alext_12m',
 'atm_frec',
 'cre_salvig_tc_rccsf_m02',
 'prm_camptot06m',
 'cnt_meses_sinegresos_12m',
 'var_usotcrrstsf03m',
 

In [29]:
import pandas as pd
import numpy as np

# ==========================================
# 1. Copiar DF original
# ==========================================
df_2 = df.copy()

# ==========================================
# 3. Columnas Int32 → rellenar NA → convertir a int64
# ==========================================
int32_cols = df_2.select_dtypes(include=["Int32"]).columns

df_2[int32_cols] = df_2[int32_cols].fillna(0).astype("int64")

# ==========================================
# 4. Columnas float → rellenar NA con 0
# ==========================================
float_cols = df_2.select_dtypes(include=["float64", "Float64"]).columns

df_2[float_cols] = df_2[float_cols].fillna(0)

# ==========================================
# 5. Columnas boolean → rellenar NA con False
# ==========================================
bool_cols = df_2.select_dtypes(include=["boolean"]).columns

df_2[bool_cols] = df_2[bool_cols].fillna(False)

# ==========================================
# 6. Columnas categóricas (strings) → NA = "SIN_INFO"
# ==========================================
cat_cols = df_2.select_dtypes(include=["object", "string"]).columns

df_2[cat_cols] = df_2[cat_cols].fillna("SIN_INFO")

In [30]:
df_2["cnt_alerta_hist"] = (
    df_2["cnt_alerta_hist"]
    .astype("float64")
)

In [31]:
df_2.tipo_alerta_n2.value_counts()

tipo_alerta_n2
SIN_INFO           237386
AUTOMATICA            539
SEMI AUTOMATICA       108
MANUAL                 32
Name: count, dtype: Int64

In [32]:
# Renombrar y reordenar columnas
#df_2 = df_2.rename(columns={'target_m': 'target'})
#df_2 =df_2[['target'] + [c for c in df_2.columns if c != 'target']]


In [33]:
df_2["tipo_alerta_n2"] =df_2["tipo_alerta_n2"].fillna(0)

In [34]:
# Eliminar columnas 'num_documento' y 'mes_base' si existen
cols_a_eliminar = ["fecha_constitucion"]
df_5 = df_2.drop(columns=cols_a_eliminar, errors="ignore")


In [35]:
# Convertir explícitamente la columna problemática a string
df_5['tipo_alerta_n2'] = df_5['tipo_alerta_n2'].astype(str)

In [36]:
df_6= df_5[(df_5.cod_mes=='202602')]

In [37]:
df_6.head()

,cod_mes,cod_cli,key_value,target_m,tipo_alerta_n2,mto_pas_soles,ros_por_trx_3m,alertas_por_antiguedad,edad,ratio_abonos_1m_vs_6m,bpi_trx_mon_min,sow_svep1actallsfm01,grupo_final,imp_trx_cargosefe_6m,pasivo_vs_ingresos,sow_lnep1tcrallsfm01,rat_trx_abonosefectot_3m,atm_monto,ind_lin_ing_tcr_ibk,atm_recen,desc_provincia,cnt_trx_abonospromtot_3m,cnt_alerta_hist,flg_alerta_12m,num_antiguedad,rat_trx_abonosefectot_9m,prm_usotcrrstsf03m,cod_ubigeo_cd,var_lintcrrstsf03m,lvl_edu,imp_trx_abonosefect_6m,cnt_trx_cargostot_3m,cre_pct_salvig_tc_rccsf_m03,ctd_camptot06m,desc_departamento,ratio_cargos_1m_vs_6m,avg_trx_cargostot_3m,cre_saltot_tc_rccsf_m02,cnt_meses_siningresos_12m,prom_salvig_pp_rccsf_06m,cnt_trx_sinenv_alext_12m,atm_frec,cre_salvig_tc_rccsf_m02,prm_camptot06m,cnt_meses_sinegresos_12m,var_usotcrrstsf03m,rec_camptot06m,cod_rsg_pep,flg_pep,max_camptot06m
0,202602,0009414564,CEF48C576BAA57AFD8EC47311348D992DAB88E84E5C43D0A61F753A29FFF94AC,0,SIN_INFO,1972.07,0.00,0.00,48,0.19,1,0.848,Digital,0.00,0.12,0.818,0.00,2472.400000000,0.000,6,LIMA,6.67,0.00,0,21,0.00,0.396,150140,0.000,SIN_INFO,0.00,44,0.335,11,LIMA,0.00,0.00,0.488,0,0.000,0,2,0.488,1.833,0,-0.567,6,0,0,3
1,202602,0015184133,8EAFC45206B8420E139253386B72F3AE4D09795611EAC129F2CA212607284F8D,0,SIN_INFO,20927.81,0.00,0.00,42,0.00,1,0.691,Digital,1150.00,0.52,0.528,0.00,700.000000000,0.000,1,LIMA,1.33,0.00,0,8,0.00,0.759,150140,0.000,SECUNDARIA COMPLETA,0.00,18,0.002,31,LIMA,6145.47,0.33,0.009,0,20528.903,0,1,0.009,5.167,0,-0.008,6,0,1,7
2,202602,0014786649,DB4BF756C539FAFA8BB074A91CE9CC6AD891F782890E9E63C274D5FA79917F67,0,SIN_INFO,4207.36,0.00,0.00,34,1.00,1,1.000,Digital,50.00,0.04,0.000,0.00,470.000000000,0.000,1,LIMA,10.00,0.00,0,9,0.00,0.000,150135,0.000,SECUNDARIA COMPLETA,0.00,111,0.000,25,LIMA,71576.23,0.00,0.000,0,0.000,0,4,0.000,4.167,0,0.000,6,0,1,5
3,202602,0014607544,294871FE43C1D3E5154E31B1AD7E21652EE6A0D27EC2DA152F910F7FC379C728,0,SIN_INFO,2615.07,0.00,0.00,33,0.10,2,0.735,Digital,1900.00,0.09,0.302,0.43,26546.360000000,0.000,2,LIMA,4.67,0.00,0,9,0.28,0.209,150113,0.002,SUPERIOR 2º AÑO,13299.45,20,0.000,22,LIMA,5923.06,0.00,0.011,0,37277.890,0,9,0.011,3.667,0,0.735,6,0,0,5
4,202602,0011399744,7A7C85FCF72CC3379E3560DA10BD2EA29771C932A91B323FC76B179898CA8414,0,SIN_INFO,1591.06,0.00,0.00,78,0.44,1,0.000,Digital,8279.20,0.04,0.493,0.17,32400.000000000,0.000,1,LIMA,2.00,0.00,0,16,0.25,0.018,150140,0.002,SUPERIOR COMPLETA,15980.00,43,-1.000,4,LIMA,4019.19,1.33,SIN_INFO,0,0.000,0,6,SIN_INFO,2.000,0,1.000,3,0,0,3


In [46]:
# Construir la lista de columnas numéricas usando el archivo de equivalencias
from pathlib import Path
# Lista de nombres iniciales que creemos que deberían ser numéricas (nombres fuente)
nombres_iniciales = [
    'ingreso_bruto', 'pasivo_soles', 'trx_monto_abonos_6m_efectivo',
    'trx_monto_cargos_6m_efectivo', 'facturacion', 'cp_promedio_mensual_ing',
    'monto_al_exterior_12m', 'edad', 'antiguedad', 'prm_lintcrallsf12m','prom_lin_tc_rccsf_06m','lin_tcrrstsf03m','prm_usotcrrstsf03m',
    'prm_lintcrrstsf03m','cre_saltot_tc_rccsf_m02','prom_salvig_entprinc_tc_rccsf_03m','cre_salvig_tc_rccsf_m02','ind_max_salvig_tc_rccsf_06m',
    'var_usotcrrstsf03m','ind_min_salvig_tc_rccsf_06m','lintot_tc_rccsf_03m','prom_salvig_pp_rccsf_06m','salvig_pp_rccsf_06m',
    'cre_pct_salvig_tc_rccsf_m03','prom_salvig_tc_rccsf_06m','cre_lin_tc_rccsf_m02','var_lintcrrstsf03m','ing_brt','ind_lin_ing_tcr_ibk',
    'sow_lnep1tcrallsfm01','sow_svep1actallsfm01','prm_camptot06m','atm_monto','atm_trx_prom','atm_trx_ret_prom',
    'atm_trx_dep_prom','atm_trx_nmon_prom','atm_trx_nmon_prom2','atm_trx_con_prom','bpi_monto','bpi_trx_prom',
    'bpi_trx_prom2','bpi_trx_nmon_prom','bpi_trx_nmon_prom2','bpi_trx_mon_prom','bpi_trx_mon_prom2','cod_ubigeo_cd'
    # ... agrega más si lo necesitas
]
# Leer archivo de equivalencias y mapear a nombres finales (si existe)
eq_path = Path(r'/equivalencias_variables.csv')
if eq_path.exists():
    eq_df = pd.read_csv(eq_path).rename(columns={'VARIABLE_INICIAL':'inicial','VARIABLE_FINAL':'final'})
    # Crear diccionario de mapeo: inicial -> final
    mapeo = dict(zip(eq_df['inicial'].astype(str), eq_df['final'].astype(str)))
    # Construir lista de columnas finales a partir de nombres_iniciales
    columnas_numericas = [mapeo.get(n, n) for n in nombres_iniciales]
else:
    # Si no hay archivo de equivalencias, usar los nombres iniciales tal cual
    columnas_numericas = nombres_iniciales

# Filtrar solo las columnas que realmente existen en df_5 (evita KeyError)
columnas_numericas = [c for c in columnas_numericas if c in df_5.columns]
if not columnas_numericas:
    print('Advertencia: no se encontraron columnas numéricas candidatas en df_5')
else:
    # Convertir a numérico con coerce para limpiar valores no numéricos
    df_6[columnas_numericas] = df_6[columnas_numericas].apply(pd.to_numeric, errors='coerce')
    # Forzar float64 (opcional) — se ignoran columnas donde no aplica
    df_6[columnas_numericas] = df_6[columnas_numericas].astype('float64', errors='ignore')

In [49]:
df_6.to_parquet(
    's3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PN/RENTA_ALTA/DATA_INFERENCIA_PILOTO/202602.parquet',
    index=False
)

In [47]:
df_6.cod_ubigeo_cd.dtypes

dtype('float64')

In [48]:
df_6.cod_ubigeo_cd.value_counts()

cod_ubigeo_cd
150140.00    24640
150122.00    12939
150130.00    12082
150114.00    12012
150131.00     9610
             ...  
120602.00        1
220405.00        1
220605.00        1
160102.00        1
150721.00        1
Name: count, Length: 1035, dtype: int64